# 제주 특산물 가격 예측 - 앙상블 모델

| 항목 | 내용 |
|------|------|
| **버전** | v3.0.0 |
| **날짜** | 2026-03-12 |
| **모델** | DNN (Keras) + CatBoost + XGBoost 앙상블 |
| **전처리** | v1.0.1 동일 (DACON 1위 전처리) |
| **앙상블** | 검증 성능 기반 가중 평균 |
| **출력** | results/submission_v3.0.0.csv |

## 모델별 특징
| 모델 | 피처 | 비고 |
|------|------|------|
| DNN | get_dummies 인코딩 | v1.0.1 동일 구조 |
| CatBoost | 원본 범주형 직접 사용 | cat_features 지정 |
| XGBoost | get_dummies 인코딩 | - |

---
## 1. 라이브러리 로드

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, datetime, random, os, platform
warnings.filterwarnings('ignore')

import holidays

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

from catboost import CatBoostRegressor
import xgboost as xgb

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.optimize import minimize

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 2024
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
tf.random.set_seed(SEED)

print(f'TF: {tf.__version__}')
print(f'CatBoost: {__import__("catboost").__version__}')
print(f'XGBoost: {xgb.__version__}')

TF: 2.20.0
CatBoost: 1.2.10
XGBoost: 3.2.0


---
## 2. 데이터 로드

In [14]:
DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')
print(f'train: {train.shape}, test: {test.shape}')
train.head(3)

train: (59397, 7), test: (1092, 5)


,ID,timestamp,item,corporation,location,supply(kg),price(원/kg)
0,TG_A_J_20190101,2019-01-01,TG,A,J,0.0,0.0
1,TG_A_J_20190102,2019-01-02,TG,A,J,0.0,0.0
2,TG_A_J_20190103,2019-01-03,TG,A,J,60601.0,1728.0


---
## 3. 전처리 (v1.0.1 동일)

In [15]:
def pre_all(train, test):
    train['timestamp'] = pd.to_datetime(train['timestamp'])
    test['timestamp']  = pd.to_datetime(test['timestamp'])
    df = pd.concat([train, test]).reset_index(drop=True)
    df.rename(columns={'supply(kg)': 'supply', 'price(원/kg)': 'price'}, inplace=True)

    df['year']     = df['timestamp'].dt.year
    df['month']    = df['timestamp'].dt.month
    df['day']      = df['timestamp'].dt.day
    df['week_day'] = df['timestamp'].dt.weekday

    le = LabelEncoder()
    df['year_month'] = df['timestamp'].map(lambda x: f'{x.year}-{x.month}')
    df['year_month'] = le.fit_transform(df['year_month'])

    df['week'] = df['timestamp'].map(
        lambda x: datetime.datetime(x.year, x.month, x.day).isocalendar()[1]
    )
    week_offsets = {2019: 0, 2020: 52, 2021: 52+53, 2022: 52+53+53, 2023: 52+53+53+52}
    df['week_num'] = df.apply(lambda r: int(r['week']) + week_offsets.get(r['year'], 0), axis=1)
    df.loc[df['timestamp'] == '2019-12-30', 'week_num'] = 52
    df.loc[df['timestamp'] == '2019-12-31', 'week_num'] = 52

    kr_holi = holidays.KR()
    df['holiday'] = df['timestamp'].map(lambda x: 1 if x in kr_holi else 0)

    train_out = df[~df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    test_out  = df[ df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    print(f'전처리 완료 — train: {train_out.shape}, test: {test_out.shape}')
    return train_out, test_out

train_pre, test_pre = pre_all(train, test)

전처리 완료 — train: (59397, 15), test: (1092, 15)


---
## 4. 이상치 처리 + 타겟 변환 + TG 공휴일 보정

In [16]:
# 이상치 처리
outlier_thresholds = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}
for item, thr in outlier_thresholds.items():
    idx = train_pre[(train_pre['item'] == item) & (train_pre['price'] > thr)].index
    if len(idx):
        mean_p = train_pre[(train_pre['item'] == item) & (train_pre['price'] != 0)]['price'].mean()
        train_pre.loc[idx, 'price'] = mean_p
        print(f'{item}: {len(idx)}개 이상치 → 평균({mean_p:.0f})')

# 타겟 변환
train_pre['is_tg'] = (train_pre['item'] == 'TG').astype(int)
test_pre['is_tg']  = (test_pre['item']  == 'TG').astype(int)
train_pre['price_transformed'] = np.where(
    train_pre['is_tg'] == 1,
    np.sqrt(train_pre['price']),
    np.log1p(train_pre['price'])
)

# TG 공휴일 보정
drop_cols_train = ['supply', 'timestamp', 'week']
drop_cols_test  = ['supply', 'timestamp', 'week', 'price']

tg_mask = (train_pre['item'] == 'TG') & (train_pre['holiday'] == 1) & (train_pre['price'] != 0)
active_holi = list(train_pre[tg_mask].groupby('timestamp').count().reset_index()['timestamp'])
fix_idx = train_pre[train_pre['timestamp'].isin(active_holi)].index
train_pre.loc[fix_idx, 'holiday'] = 0
print(f'TG 공휴일 보정: {len(fix_idx)}개')

TG: 1개 이상치 → 평균(4145)
RD: 1개 이상치 → 평균(564)
BC: 1개 이상치 → 평균(2757)
CB: 7개 이상치 → 평균(715)
TG 공휴일 보정: 1521개


---
## 5. 피처 인코딩 (DNN / XGBoost용 get_dummies)

In [17]:
# DNN / XGBoost: get_dummies 인코딩
Xy_all = pd.get_dummies(
    train_pre.drop(columns=drop_cols_train),
    columns=['item', 'corporation', 'location']
).sort_values('ID').reset_index(drop=True)

test_enc = pd.get_dummies(
    test_pre.drop(columns=drop_cols_test),
    columns=['item', 'corporation', 'location']
).sort_values('ID').reset_index(drop=True)

# 컬럼 정렬
for col in Xy_all.columns:
    if col not in test_enc.columns:
        test_enc[col] = 0

ID_COL     = 'ID'
TARGET_COL = 'price'
TARGET_TRF = 'price_transformed'
FEAT_COLS  = [c for c in Xy_all.columns if c not in [ID_COL, TARGET_COL, TARGET_TRF, 'is_tg']]

print(f'FEAT_COLS: {len(FEAT_COLS)}개')
print(FEAT_COLS)

FEAT_COLS: 20개
['year', 'month', 'day', 'week_day', 'year_month', 'week_num', 'holiday', 'item_BC', 'item_CB', 'item_CR', 'item_RD', 'item_TG', 'corporation_A', 'corporation_B', 'corporation_C', 'corporation_D', 'corporation_E', 'corporation_F', 'location_J', 'location_S']


---
## 6. 학습/검증 분리 (시간순 8:2)

In [18]:
Xy_sorted = Xy_all.sort_values('year_month').reset_index(drop=True)
split_idx = int(len(Xy_sorted) * 0.8)
train_df = Xy_sorted.iloc[:split_idx].copy()
val_df   = Xy_sorted.iloc[split_idx:].copy()

X_tr = train_df[FEAT_COLS].values
y_tr = train_df[TARGET_TRF].values
X_vl = val_df[FEAT_COLS].values
y_vl = val_df[TARGET_TRF].values

y_vl_orig  = val_df[TARGET_COL].values
y_vl_is_tg = val_df.get('item_TG', pd.Series([False]*len(val_df))).values
X_test     = test_enc[FEAT_COLS].values
test_is_tg = test_enc.get('item_TG', pd.Series([False]*len(test_enc))).values.astype(bool)

print(f'학습: {X_tr.shape}, 검증: {X_vl.shape}')

학습: (47517, 20), 검증: (11880, 20)


---
## 7. 모델 1 — DNN (Keras)

> v1.0.1과 동일한 구조

In [19]:
# 스케일링
dnn_scaler = StandardScaler()
X_tr_sc = dnn_scaler.fit_transform(X_tr)
X_vl_sc = dnn_scaler.transform(X_vl)
X_test_sc = dnn_scaler.transform(X_test)

# 모델 정의
def build_dnn(input_dim, lr=1e-3):
    l2 = regularizers.l2(1e-4)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, kernel_regularizer=l2),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(128, kernel_regularizer=l2),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.2),
        layers.Dense(64,  kernel_regularizer=l2),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.1),
        layers.Dense(32,  kernel_regularizer=l2),
        layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dense(1)
    ], name='dnn_v3')
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='mae', metrics=['mse'])
    return model

os.makedirs('./models',  exist_ok=True)
os.makedirs('./results', exist_ok=True)

dnn_model = build_dnn(len(FEAT_COLS))

dnn_cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=20,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=8, min_lr=1e-6, verbose=0),
    callbacks.ModelCheckpoint('./models/dnn_v3.0.0.keras',
                              monitor='val_loss', save_best_only=True, verbose=0)
]

dnn_history = dnn_model.fit(
    X_tr_sc, y_tr,
    validation_data=(X_vl_sc, y_vl),
    epochs=200, batch_size=512,
    callbacks=dnn_cb, verbose=1
)

dnn_best_epoch = np.argmin(dnn_history.history['val_loss'])
print(f'DNN Best Epoch: {dnn_best_epoch}')

Epoch 1/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 12.1082 - mse: 645.4938 - val_loss: 12.6616 - val_mse: 674.3748 - learning_rate: 0.0010
Epoch 2/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 10.2508 - mse: 489.5679 - val_loss: 11.3033 - val_mse: 552.1913 - learning_rate: 0.0010
Epoch 3/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 8.3758 - mse: 348.3018 - val_loss: 8.7644 - val_mse: 345.5581 - learning_rate: 0.0010
Epoch 4/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 6.5392 - mse: 225.4251 - val_loss: 7.9200 - val_mse: 281.4908 - learning_rate: 0.0010
Epoch 5/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 5.1178 - mse: 142.2885 - val_loss: 6.1485 - val_mse: 178.4458 - learning_rate: 0.0010
Epoch 6/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 4.1774 - mse: 108.5132 - val_loss: 5.7489 - val_mse: 157.8624 - learning_rate: 0.0010
Epoch 7/200
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 3.7545 - mse: 103.3148 - val_loss: 4.6553 - val_mse: 127.160

---
## 8. 모델 2 — CatBoost

> 원본 범주형 컬럼(`item`, `corporation`, `location`) 직접 사용

In [20]:
# CatBoost용 데이터 (원본 범주형 컬럼 활용)
CAT_COLS  = ['item', 'corporation', 'location']
TIME_COLS = ['year_month', 'week_num', 'week_day', 'month', 'day', 'holiday', 'year']
CB_FEAT   = TIME_COLS + CAT_COLS

# 시간순 분리 (train_pre / test_pre 기반)
tr_pre_s   = train_pre.sort_values('year_month').reset_index(drop=True)
sp          = int(len(tr_pre_s) * 0.8)
cb_train    = tr_pre_s.iloc[:sp]
cb_val      = tr_pre_s.iloc[sp:]

X_cb_tr   = cb_train[CB_FEAT]
y_cb_tr   = cb_train[TARGET_TRF].values
X_cb_vl   = cb_val[CB_FEAT]
y_cb_vl   = cb_val[TARGET_TRF].values
X_cb_test = test_pre[CB_FEAT]

cb_is_tg_vl   = (cb_val['item'] == 'TG').values
cb_vl_orig    = cb_val[TARGET_COL].values
cb_test_is_tg = (test_pre['item'] == 'TG').values

print(f'CatBoost 학습: {X_cb_tr.shape}, 검증: {X_cb_vl.shape}')

CatBoost 학습: (47517, 10), 검증: (11880, 10)


In [ ]:
cb_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.05,
    depth=6,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=CAT_COLS,
    random_seed=SEED,
    early_stopping_rounds=100,
    verbose=200
)

cb_model.fit(
    X_cb_tr, y_cb_tr,
    eval_set=(X_cb_vl, y_cb_vl),
    use_best_model=True
)

print(f'CatBoost 최적 반복: {cb_model.get_best_iteration()}')

0:	learn: 12.5446566	test: 14.6777836	best: 14.6777836 (0)	total: 102ms	remaining: 5m 7s
200:	learn: 3.2251291	test: 3.8734031	best: 3.8734031 (200)	total: 15s	remaining: 3m 29s
400:	learn: 3.0409827	test: 3.7510998	best: 3.7510647 (378)	total: 29.9s	remaining: 3m 14s
600:	learn: 2.9745787	test: 3.7214432	best: 3.7213959 (597)	total: 45.1s	remaining: 2m 59s


KeyboardInterrupt: 

---
## 9. 모델 3 — XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:absoluteerror',
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=100
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_vl, y_vl)],
    verbose=200
)

print(f'XGB 최적 반복: {xgb_model.best_iteration}')

---
## 10. 개별 모델 검증 성능 비교

In [ ]:
def inverse_transform(y_transformed, is_tg):
    """price_transformed → 원본 가격"""
    result = np.where(
        is_tg.astype(bool),
        np.power(np.clip(y_transformed, 0, None), 2),
        np.expm1(y_transformed)
    )
    return np.clip(result, 0, None)


def eval_model(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    print(f'  [{name}]  MAE={mae:>8,.1f}  RMSE={rmse:>8,.1f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return mae, rmse, r2, mape


# DNN 검증
dnn_vl_pred_trf = dnn_model.predict(X_vl_sc, verbose=0).flatten()
dnn_vl_pred     = inverse_transform(dnn_vl_pred_trf, y_vl_is_tg)

# CatBoost 검증 (val_df 기준으로 정렬 맞춤)
cb_vl_pred_trf  = cb_model.predict(X_cb_vl)
cb_vl_pred      = inverse_transform(cb_vl_pred_trf, cb_is_tg_vl)

# XGBoost 검증
xgb_vl_pred_trf = xgb_model.predict(X_vl)
xgb_vl_pred     = inverse_transform(xgb_vl_pred_trf, y_vl_is_tg)

print('=' * 70)
print('  개별 모델 검증 성능 (가격 원복 후)')
print('=' * 70)
dnn_scores = eval_model(y_vl_orig,   dnn_vl_pred,  'DNN     ')
cb_scores  = eval_model(cb_vl_orig,  cb_vl_pred,   'CatBoost')
xgb_scores = eval_model(y_vl_orig,   xgb_vl_pred,  'XGBoost ')
print('=' * 70)

---
## 11. 앙상블

> 검증 MAE 역수 비례 가중치 + scipy 최적화 두 가지 비교

In [ ]:
# CatBoost 예측을 val_df 인덱스로 정렬
# (CatBoost은 cb_val 기반, DNN/XGB는 val_df 기반 → 같은 ID 순서 확인)
# 안전하게: val_df ID와 cb_val ID 비교
assert (val_df['ID'].values == cb_val['ID'].values).all(), \
    'val_df와 cb_val의 ID 순서가 다릅니다 — 확인 필요'

# ── 방법 1: MAE 역수 비례 가중치 ──────────────────────────────
mae_dnn, mae_cb, mae_xgb = dnn_scores[0], cb_scores[0], xgb_scores[0]
inv_mae = np.array([1/mae_dnn, 1/mae_cb, 1/mae_xgb])
w_inv   = inv_mae / inv_mae.sum()
print(f'MAE 역수 가중치  DNN={w_inv[0]:.3f}  CB={w_inv[1]:.3f}  XGB={w_inv[2]:.3f}')

ens_pred_inv = (w_inv[0] * dnn_vl_pred
              + w_inv[1] * cb_vl_pred
              + w_inv[2] * xgb_vl_pred)
mae_ens_inv = mean_absolute_error(y_vl_orig, ens_pred_inv)
print(f'앙상블 MAE (역수 가중): {mae_ens_inv:,.2f}')

# ── 방법 2: scipy 최적화 ────────────────────────────────────────
def ensemble_mae(w):
    w = np.abs(w) / np.abs(w).sum()
    pred = w[0]*dnn_vl_pred + w[1]*cb_vl_pred + w[2]*xgb_vl_pred
    return mean_absolute_error(y_vl_orig, pred)

res = minimize(ensemble_mae, x0=[1/3, 1/3, 1/3],
               method='Nelder-Mead',
               options={'maxiter': 1000, 'xatol': 1e-5})
w_opt = np.abs(res.x) / np.abs(res.x).sum()
print(f'\n최적화 가중치    DNN={w_opt[0]:.3f}  CB={w_opt[1]:.3f}  XGB={w_opt[2]:.3f}')

ens_pred_opt = (w_opt[0] * dnn_vl_pred
              + w_opt[1] * cb_vl_pred
              + w_opt[2] * xgb_vl_pred)
mae_ens_opt = mean_absolute_error(y_vl_orig, ens_pred_opt)
print(f'앙상블 MAE (최적화)  : {mae_ens_opt:,.2f}')

# 더 좋은 가중치 선택
if mae_ens_opt <= mae_ens_inv:
    w_final = w_opt
    print('\n→ 최적화 가중치 사용')
else:
    w_final = w_inv
    print('\n→ 역수 가중치 사용')

ens_vl_pred = (w_final[0] * dnn_vl_pred
             + w_final[1] * cb_vl_pred
             + w_final[2] * xgb_vl_pred)

In [ ]:
mae  = mean_absolute_error(y_vl_orig, ens_vl_pred)
rmse = np.sqrt(mean_squared_error(y_vl_orig, ens_vl_pred))
r2   = r2_score(y_vl_orig, ens_vl_pred)
mape = np.mean(np.abs((y_vl_orig - ens_vl_pred) / (y_vl_orig + 1e-8))) * 100

print('=' * 60)
print('  앙상블 v3.0.0 최종 검증 성능')
print('=' * 60)
print(f'  가중치  DNN={w_final[0]:.3f}  CB={w_final[1]:.3f}  XGB={w_final[2]:.3f}')
print(f'  MAE  : {mae:>10,.2f} 원/kg')
print(f'  RMSE : {rmse:>10,.2f} 원/kg')
print(f'  R²   : {r2:>10.4f}')
print(f'  MAPE : {mape:>10.2f} %')
print('=' * 60)

In [ ]:
# 모델별 비교 바 차트
model_names = ['DNN', 'CatBoost', 'XGBoost', 'Ensemble']
maes = [dnn_scores[0], cb_scores[0], xgb_scores[0], mae]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue', 'darkorange', 'seagreen', 'crimson']
bars = axes[0].bar(model_names, maes, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title('모델별 검증 MAE', fontsize=13)
axes[0].set_ylabel('MAE (원/kg)')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=10)

max_v = max(y_vl_orig.max(), ens_vl_pred.max())
axes[1].scatter(y_vl_orig, ens_vl_pred, alpha=0.3, s=8, color='crimson')
axes[1].plot([0, max_v], [0, max_v], 'k--', lw=2)
axes[1].set_title(f'앙상블 실제 vs 예측  R²={r2:.4f}', fontsize=12)
axes[1].set_xlabel('실제'); axes[1].set_ylabel('예측'); axes[1].grid(alpha=0.3)

plt.suptitle('앙상블 v3.0.0 검증 결과', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/ensemble_eval_v3.0.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. 테스트 예측 + 후처리 + 제출

In [ ]:
# 각 모델 테스트 예측 (변환 공간)
dnn_test_trf  = dnn_model.predict(X_test_sc, verbose=0).flatten()
cb_test_trf   = cb_model.predict(X_cb_test)
xgb_test_trf  = xgb_model.predict(X_test)

# 원본 가격으로 역변환
dnn_test_pred  = inverse_transform(dnn_test_trf,  test_is_tg)
cb_test_pred   = inverse_transform(cb_test_trf,   cb_test_is_tg)
xgb_test_pred  = inverse_transform(xgb_test_trf,  test_is_tg)

# 앙상블
ens_test_pred = (w_final[0] * dnn_test_pred
               + w_final[1] * cb_test_pred
               + w_final[2] * xgb_test_pred)
ens_test_pred = np.clip(ens_test_pred, 0, None)

# 후처리: 품목별 최솟값 미만 → 0
test_result_df = test_pre[['ID', 'item']].copy()
test_result_df['answer'] = ens_test_pred

min_thresholds = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for item, thr in min_thresholds.items():
    mask = (test_result_df['item'] == item) & (test_result_df['answer'] < thr)
    test_result_df.loc[mask, 'answer'] = 0.0
    n = mask.sum()
    if n > 0:
        print(f'{item}: {n}개 → 0 처리 (임계값: {thr})')

print('\n예측 통계:')
print(test_result_df.groupby('item')['answer'].agg(['mean','min','max']).round(1))

In [ ]:
result = sub[['ID']].merge(test_result_df[['ID', 'answer']], on='ID', how='left')
result['answer'] = result['answer'].fillna(0.0)

SUBMISSION_PATH = './results/submission_v3.0.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')
print(f'저장: {SUBMISSION_PATH}, 행: {len(result)}')
result.head(10)

---
## 13. 결과 요약

In [ ]:
print('=' * 65)
print('  앙상블 v3.0.0 최종 결과')
print('=' * 65)
print(f'  가중치  DNN={w_final[0]:.3f}  CatBoost={w_final[1]:.3f}  XGBoost={w_final[2]:.3f}')
print()
print('  [개별 모델 검증 MAE]')
print(f'    DNN      : {dnn_scores[0]:>10,.2f} 원/kg')
print(f'    CatBoost : {cb_scores[0]:>10,.2f} 원/kg')
print(f'    XGBoost  : {xgb_scores[0]:>10,.2f} 원/kg')
print()
print('  [앙상블 검증 성능]')
print(f'    MAE  = {mae:>10,.2f} 원/kg')
print(f'    RMSE = {rmse:>10,.2f} 원/kg')
print(f'    R²   = {r2:>10.4f}')
print(f'    MAPE = {mape:>10.2f} %')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print('=' * 65)

### 다음 버전 개선 방향

| 버전 | 개선 내용 |
|------|----------|
| **v3.1.0** | LightGBM 추가 (4모델 앙상블) |
| **v3.2.0** | Stacking (메타 러너) |
| **v3.3.0** | LSTM(v2) + 트리 모델 앙상블 |
| **v4.0.0** | Optuna 하이퍼파라미터 최적화 |